# Visualização Avançada de Dados

Até agora aprendemos a explorar e entender os dados usando pandas. Mas uma boa análise não é completa sem uma boa visualização.Neste notebook vamos aprender a criar gráficos de diferentes tipos usando duas das bibliotecas mais populares do Python:

- **Seaborn**: focado em gráficos estatísticos bonitos e informativos, construído sobre o Matplotlib
- **Plotly**: gráficos interativos (você pode passar o mouse, dar zoom, filtrar)

Vamos continuar usando o dataset da Netflix.

## Importando as Bibliotecas

Antes de tudo, precisamos importar as bibliotecas que vamos usar neste notebook.
> Se aparecer algum erro de importação, lembre-se de instalar a biblioteca com `pip install <nome>`.

In [ ]:
%pip install pandas
%pip install numpy
%pip install seaborn
%pip install matplotlib.pyplot
%pip install plotly.express
%pip install plotly.graph_objects

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Configurações visuais do Seaborn
sns.set_theme(style="darkgrid", palette="muted")

print("✅ Bibliotecas carregadas com sucesso!")

## Carregando e Preparando os Dados

Vamos carregar o dataset da Netflix e fazer algumas preparações básicas para que possamos criar nossos gráficos.

In [ ]:
# Carregando o dataset
df = pd.read_csv("data/03_netflix.csv")

# Preparações básicas (baseado no que aprendemos no notebook 04)
df["type"] = df["type"].replace({"Show": "TV Show", "Tow": "TV Show", "Tw": "TV Show", "MV": "Movie"})
df["date_added"] = pd.to_datetime(df["date_added"])
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month

# Extraindo duração numérica para filmes
movies = df[df["type"] == "Movie"].copy()
movies["duration_min"] = movies["duration"].str.extract(r"(\d+)").astype(float)

print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas")
df.head()

---

# 1. Gráfico de Barras (Bar Chart)

O gráfico de barras é um dos mais usados em análise de dados. Ele é ideal para **comparar quantidades entre categorias**.

**Quando usar:** quando você quer comparar grupos diferentes entre si (ex: filmes vs séries, países, anos).

Vamos ver quantos filmes e séries existem no catálogo da Netflix.

In [ ]:
# --- Seaborn: Gráfico de Barras ---
contagem = df["type"].value_counts().reset_index()
contagem.columns = ["Tipo", "Quantidade"]

plt.figure(figsize=(7, 5))
sns.barplot(data=contagem, x="Tipo", y="Quantidade", palette="Set2")
plt.title("Filmes vs Séries na Netflix", fontsize=14, fontweight="bold")
plt.xlabel("Tipo de Conteúdo")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly: Gráfico de Barras Interativo ---
fig = px.bar(
    contagem,
    x="Tipo",
    y="Quantidade",
    color="Tipo",
    title="Filmes vs Séries na Netflix (Interativo)",
    labels={"Tipo": "Tipo de Conteúdo", "Quantidade": "Quantidade"},
    text_auto=True
)
fig.update_layout(showlegend=False)
fig.show()

> 💡 **Dica:** No gráfico do Plotly você pode passar o mouse em cima das barras para ver os valores e dar zoom 

**EXERCÍCIO:**
Crie um gráfico de barras mostrando os **10 países com mais conteúdo** no catálogo da Netflix.

Dica: use `df["country"].value_counts().head(10)` para calcular.

In [ ]:
paises = df["country"].value_counts().head(10).reset_index()
paises_dataframe = paises

In [ ]:
# Sua resposta
plt.figure(figsize=(10, 5))
sns.barplot(data=paises, x="country", y="count", palette="Set2")
plt.title("10 Países com mais conteúdo na Netflix", fontsize=14, fontweight="bold")
plt.xlabel("Países")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()


---

# 2. Histograma (Histogram)

O [histograma](https://pt.khanacademy.org/math/statistics-probability/displaying-describing-data/quantitative-data-graphs/v/histograms) é muito parecido com o gráfico de barras, mas tem um propósito diferente: ele mostra a **distribuição de uma variável numérica contínua**.
Em vez de comparar categorias, ele divide os valores em "faixas" (chamadas de bins) e mostra quantos registros caem em cada faixa.

**Quando usar:** para entender como os dados estão distribuídos (ex: duração dos filmes, idades, preços).

In [ ]:
# --- Seaborn: Histograma da duração dos filmes ---
plt.figure(figsize=(10, 5))
sns.histplot(data=movies, x="duration_min", bins=40, kde=True, color="steelblue")
plt.title("Distribuição da Duração dos Filmes (em minutos)", fontsize=14, fontweight="bold")
plt.xlabel("Duração (minutos)")
plt.ylabel("Quantidade de Filmes")
plt.axvline(movies["duration_min"].mean(), color="red", linestyle="--", label=f"Média: {movies['duration_min'].mean():.0f} min")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Duração média dos filmes: {movies['duration_min'].mean():.0f} minutos")
print(f"Duração mínima: {movies['duration_min'].min():.0f} minutos")
print(f"Duração máxima: {movies['duration_min'].max():.0f} minutos")

In [ ]:
# --- Plotly: Histograma Interativo ---
fig = px.histogram(
    movies,
    x="duration_min",
    nbins=40,
    title="Distribuição da Duração dos Filmes (Interativo)",
    labels={"duration_min": "Duração (minutos)"},
    color_discrete_sequence=["#636EFA"]
)
fig.update_layout(bargap=0.05)
fig.show()

> 💡 **Curva KDE (linha suave):** O parâmetro `kde=True` no Seaborn adiciona uma curva de densidade suavizada sobre o histograma. Ela mostra a "forma" da distribuição de forma mais limpa.

**EXERCÍCIO:**
Observe o histograma. A distribuição é simétrica ou assimétrica? A maioria dos filmes tem menos ou mais de 2 horas?

Sua resposta: A distribuição é relativamente simétrica, e a maioria dos filmes tem menos de 2 horas de duração.

---

# 3. Boxplot

O [boxplot](https://pt.khanacademy.org/math/statistics-probability/summarizing-quantitative-data/box-whisker-plots/v/constructing-a-box-and-whisker-plot) (ou diagrama de caixa) é um dos gráficos mais poderosos para **visualizar a distribuição e identificar outliers** (valores muito diferentes dos demais).

Ele resume em um único gráfico:
- O valor **mínimo** e **máximo**
- O **primeiro quartil** (Q1 — 25% dos dados estão abaixo)
- A **mediana** (Q2 — metade dos dados estão acima, metade abaixo)
- O **terceiro quartil** (Q3 — 75% dos dados estão abaixo)
- Os **[outliers](https://pt.khanacademy.org/math/statistics-probability/summarizing-quantitative-data/box-whisker-plots/v/judging-outliers-in-a-dataset)** (pontos fora dos "bigodes")

**Quando usar:** para comparar a distribuição entre grupos ou identificar valores extremos.

In [ ]:
# --- Seaborn: Boxplot da duração dos filmes por rating ---
# Filtrando ratings mais comuns para não poluir o gráfico
ratings_comuns = movies["rating"].value_counts().head(6).index
movies_filtrados = movies[movies["rating"].isin(ratings_comuns)]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=movies_filtrados,
    x="rating",
    y="duration_min",
    palette="Set3",
    order=ratings_comuns
)
plt.title("Duração dos Filmes por Classificação Etária", fontsize=14, fontweight="bold")
plt.xlabel("Classificação Etária (Rating)")
plt.ylabel("Duração (minutos)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly: Boxplot Interativo ---
fig = px.box(
    movies_filtrados,
    x="rating",
    y="duration_min",
    color="rating",
    title="Duração dos Filmes por Classificação Etária (Interativo)",
    labels={"rating": "Classificação Etária", "duration_min": "Duração (minutos)"},
    points="outliers"  # mostra pontos outliers
)
fig.update_layout(showlegend=False)
fig.show()

> 💡 **Como ler o Boxplot:**
> - A **linha do meio** da caixa = mediana
> - A **caixa** vai do Q1 ao Q3 (chamado de IQR - Intervalo Interquartil)
> - As **linhas finas** (bigodes) vão até 1.5x o IQR
> - Os **pontos isolados** fora dos bigodes são os outliers!

**EXERCÍCIO:**
Olhando o boxplot, qual classificação etária tende a ter filmes mais longos? Existem outliers visíveis?

Sua resposta:

---

# 4️⃣ Gráfico de Linha (Line Chart)

O gráfico de linha é ideal para mostrar **evolução ao longo do tempo**.

**Quando usar:** para analisar tendências temporais (ex: crescimento do catálogo por ano, variação de preços ao longo do tempo).

In [ ]:
# Filmes e séries adicionados por ano
por_ano = df.groupby(["year_added", "type"]).size().reset_index(name="quantidade")
por_ano = por_ano[por_ano["year_added"].between(2010, 2021)]  # filtrando anos com mais dados

# --- Seaborn: Gráfico de Linha ---
plt.figure(figsize=(12, 5))
sns.lineplot(data=por_ano, x="year_added", y="quantidade", hue="type", marker="o", linewidth=2)
plt.title("Crescimento do Catálogo Netflix por Ano", fontsize=14, fontweight="bold")
plt.xlabel("Ano")
plt.ylabel("Quantidade de Títulos Adicionados")
plt.legend(title="Tipo")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly: Gráfico de Linha Interativo ---
fig = px.line(
    por_ano,
    x="year_added",
    y="quantidade",
    color="type",
    markers=True,
    title="Crescimento do Catálogo Netflix por Ano (Interativo)",
    labels={"year_added": "Ano", "quantidade": "Quantidade de Títulos Adicionados", "type": "Tipo"}
)
fig.show()

**EXERCÍCIO:**
Em que ano o catálogo cresceu mais rapidamente? Isso faz sentido considerando o crescimento da Netflix no mundo?

Sua resposta: No ano de 2017 o catálogo cresceu mais rapidamente. Faz sentido, já que a partir do ano de 2016, a plataforma passou a receber mais visibilidade e reconhecimento.

---

# 5. Gráfico de Pizza / Rosca (Pie / Donut Chart)

O gráfico de pizza mostra **proporções entre categorias**. É útil quando você quer mostrar como um todo se divide em partes.

> ⚠️ **Cuidado:** Gráficos de pizza são mais difíceis de comparar quando há muitas categorias. Prefira o de barras nesses casos!

**Quando usar:** quando há poucas categorias (2–5) e você quer mostrar proporção.

In [ ]:
# --- Matplotlib: Gráfico de Pizza ---
plt.figure(figsize=(7, 7))
plt.pie(
    contagem["Quantidade"],
    labels=contagem["Tipo"],
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("Set2")
)
plt.title("Proporção de Filmes vs Séries na Netflix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly: Gráfico de Pizza ---
fig = px.pie(
    contagem,
    values="Quantidade",
    names="Tipo",
    title="Proporção de Filmes vs Séries na Netflix",
    hole=0.4  # hole=0 é pizza, hole>0 é rosca (donut)
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

---

# 6. Gráfico de Dispersão (Scatter Plot)

O [scatter plot](https://pt.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/introduction-to-scatterplots/a/scatterplots-and-correlation-review) mostra a **relação entre duas variáveis numéricas**. Cada ponto representa um registro do dataset.

**Quando usar:** para verificar se existe correlação entre duas variáveis (ex: quanto maior X, maior Y?).

Vamos investigar se o ano de lançamento tem relação com a duração dos filmes na Netflix.

In [ ]:
# Filtrando filmes de 1990 em diante para uma visualização mais limpa
movies_recentes = movies[movies["release_year"] >= 1990].dropna(subset=["duration_min"])

# --- Seaborn: Scatter Plot ---
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=movies_recentes,
    x="release_year",
    y="duration_min",
    alpha=0.4,
    hue="rating",
    palette="tab10",
    s=30
)
plt.title("Relação entre Ano de Lançamento e Duração dos Filmes", fontsize=14, fontweight="bold")
plt.xlabel("Ano de Lançamento")
plt.ylabel("Duração (minutos)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Rating")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly: Scatter Plot Interativo ---
fig = px.scatter(
    movies_recentes,
    x="release_year",
    y="duration_min",
    color="rating",
    hover_name="title",
    opacity=0.5,
    title="Relação entre Ano de Lançamento e Duração dos Filmes (Interativo)",
    labels={"release_year": "Ano de Lançamento", "duration_min": "Duração (minutos)", "rating": "Classificação"}
)
fig.show()

> 💡 **No Plotly:** passe o mouse em cima dos pontos para ver o nome do filme!

**EXERCÍCIO:**
Você consegue identificar algum padrão? Os filmes mais recentes são mais longos ou mais curtos que os antigos?

Sua resposta: Sim, todos possuem classificações semelhantes, e os filmes atuais são mais curtos que os antigos.

---

# 7. Heatmap (Mapa de Calor)

O [heatmap](https://medium.com/@jessicacunha.jsc/mapa-de-calor-de-correla%C3%A7%C3%A3o-heatmap-ba37f5405400) usa **cores para representar a intensidade de valores** em uma matriz. É muito usado para visualizar correlações entre variáveis ou padrões em tabelas cruzadas.

**Quando usar:** para ver padrões entre duas variáveis categóricas ao mesmo tempo, ou para visualizar a matriz de correlação.

In [ ]:
# Conteúdo adicionado por mês e ano (tabela cruzada)
df_filtrado = df[df["year_added"].between(2015, 2021)].dropna(subset=["month_added", "year_added"])

pivot = df_filtrado.pivot_table(
    index="month_added",
    columns="year_added",
    values="show_id",
    aggfunc="count"
).fillna(0)

# Renomeando o índice para nomes dos meses
nomes_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
pivot.index = nomes_meses

# --- Seaborn: Heatmap ---
plt.figure(figsize=(12, 6))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Qtd de Títulos"}
)
plt.title("Títulos Adicionados à Netflix por Mês e Ano", fontsize=14, fontweight="bold")
plt.xlabel("Ano")
plt.ylabel("Mês")
plt.tight_layout()
plt.show()

> 💡 **Como ler o Heatmap:** Células mais escuras (vermelho/laranja) indicam mais títulos adicionados. Células mais claras (amarelo) indicam menos títulos. O número dentro de cada célula mostra o valor exato.

**EXERCÍCIO:**
Existe algum mês que a Netflix costuma adicionar mais conteúdo? Você consegue identificar algum padrão sazonal?

Sua resposta: Sim, os meses de julho, agosto e setembro, geralmente na época de recesso escolar.

---

# 8. Violin Plot

O violin plot é uma combinação do **boxplot** com a **curva de densidade (KDE)**. Ele mostra não só o resumo estatístico, mas também a "forma" da distribuição.

**Quando usar:** quando você quer mais detalhe que o boxplot, mostrando se a distribuição tem um ou mais picos.

In [ ]:
# --- Seaborn: Violin Plot ---
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=movies_filtrados,
    x="rating",
    y="duration_min",
    palette="muted",
    inner="quartile"  # mostra os quartis dentro do violino
)
plt.title("Distribuição da Duração dos Filmes por Rating (Violin Plot)", fontsize=14, fontweight="bold")
plt.xlabel("Classificação Etária")
plt.ylabel("Duração (minutos)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly: Violin Plot Interativo ---
fig = px.violin(
    movies_filtrados,
    x="rating",
    y="duration_min",
    color="rating",
    box=True,          # adiciona o boxplot dentro
    points="outliers", # mostra apenas outliers
    title="Distribuição da Duração dos Filmes por Rating (Interativo)",
    labels={"rating": "Classificação Etária", "duration_min": "Duração (minutos)"}
)
fig.update_layout(showlegend=False)
fig.show()

> 💡 **Violino vs Boxplot:** O boxplot resume a distribuição em 5 números. O violin plot mostra a forma completa. Se o violin for largo no meio, quer dizer que muitos filmes têm duração próxima àquele valor!

**DESAFIO:**
Compare o violin plot com o boxplot que criamos anteriormente (seção 3). Que informações novas o violin plot revela que o boxplot não mostrava?

Sua resposta:

---

# .9 Gráfico de Barras Empilhadas (Stacked Bar)

O gráfico de barras empilhadas é útil para mostrar tanto o **total** quanto a **composição** de cada categoria.

**Quando usar:** quando você quer comparar totais entre grupos e também ver como cada grupo se divide.

In [ ]:
# Quantos filmes e séries foram adicionados por ano?
df_filtrado_tipo = df[df["year_added"].between(2015, 2021)].dropna(subset=["year_added"])
por_ano_tipo = df_filtrado_tipo.groupby(["year_added", "type"]).size().reset_index(name="quantidade")

# --- Plotly: Barras Empilhadas ---
fig = px.bar(
    por_ano_tipo,
    x="year_added",
    y="quantidade",
    color="type",
    barmode="stack",
    title="Composição do Catálogo Netflix por Ano (Empilhado)",
    labels={"year_added": "Ano", "quantidade": "Títulos Adicionados", "type": "Tipo"},
    text_auto=True
)
fig.show()

In [ ]:
# --- Plotly: Barras Agrupadas (comparação lado a lado) ---
fig = px.bar(
    por_ano_tipo,
    x="year_added",
    y="quantidade",
    color="type",
    barmode="group",  # mudando para agrupadas
    title="Filmes vs Séries Adicionados por Ano (Agrupado)",
    labels={"year_added": "Ano", "quantidade": "Títulos Adicionados", "type": "Tipo"}
)
fig.show()

**EXERCÍCIO:**
Compare as duas versões do gráfico (empilhado vs agrupado). Quando cada um é mais útil?

Sua resposta:

---

# 10. Pair Plot — Tudo de Uma Vez!

O pair plot do Seaborn é uma ferramenta poderosa para **exploração inicial de dados numéricos**. Ele cria automaticamente uma grade com:
- **Scatter plots** entre pares de variáveis
- **Histogramas** ou distribuições na diagonal

**Quando usar:** no início de uma análise, para ter uma visão geral de como as variáveis numéricas se relacionam entre si.

In [ ]:
# Selecionando variáveis numéricas relevantes
movies_pair = movies[["duration_min", "release_year", "year_added", "type"]].dropna()
movies_pair = movies_pair[movies_pair["release_year"] >= 1990]

# --- Seaborn: Pair Plot ---
sns.pairplot(
    movies_pair,
    hue="type",
    diag_kind="kde",
    plot_kws={"alpha": 0.4}
)
plt.suptitle("Pair Plot — Relações entre Variáveis dos Filmes", y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

#  Resumo: Qual Gráfico Usar?

| Gráfico | Uso Principal | Exemplo |
|---|---|---|
| **Barras** | Comparar categorias | Filmes vs Séries |
| **Histograma** | Distribuição de variável numérica | Duração dos filmes |
| **Boxplot** | Distribuição + outliers por grupo | Duração por rating |
| **Linha** | Evolução ao longo do tempo | Crescimento do catálogo |
| **Pizza/Rosca** | Proporções (poucas categorias) | % de filmes vs séries |
| **Scatter** | Relação entre dois números | Ano vs duração |
| **Heatmap** | Padrões em tabela cruzada | Adições por mês/ano |
| **Violin** | Distribuição detalhada por grupo | Duração por rating |
| **Barras empilhadas** | Total + composição | Catálogo por ano |
| **Pair Plot** | Explorar várias variáveis de uma vez | Relações gerais |

---

## DESAFIO

Usando tudo que aprendeu neste notebook, crie uma **mini análise visual** respondendo à pergunta: **"Como o catálogo da Netflix mudou ao longo dos anos?"**

Use pelo menos **3 tipos de gráficos diferentes** e adicione textos explicando o que cada gráfico mostra.

In [ ]:
# Sua análise
